In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:
#load data
train_data = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")
test_data = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

train_data.head()
print(train_data.columns)
print(test_data.columns)

In [ ]:
#environment

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
#处理数据

def preprocessing(data):
    
    #Organize prompt and response
    def clean_text(text):
        if isinstance(text,str):
            return text.strip("[]'\\")
                              
    for col in ['prompt','response_a','response_b']:
            data[col]=data[col].apply(clean_text)
            
    data["prompt_and_response"] = "Q: "+data['prompt'].astype(str)+"ANSWER: "+data['response_a'].astype(str)+" VS "+data['response_b'].astype(str)
    
    #Add new column
    #data['winner'] = np.argmax(data[['winner_model_a', 'winner_model_b', 'winner_tie']].values, axis=1)
    
    #delete useless columns
    data = data.drop(['prompt','response_a','response_b'],axis=1)
    return data

train = preprocessing(train_data)
train['winner'] = np.argmax(train[['winner_model_a', 'winner_model_b', 'winner_tie']].values, axis=1)
train = train.drop(['winner_model_a', 'winner_model_b', 'winner_tie','model_a','model_b'],axis=1)

test = preprocessing(test_data)
print(train.head())
print(test.head())


In [ ]:
y = train['winner'] #label

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english'
)

X = vectorizer.fit_transform(train["prompt_and_response"].astype(str))


from sklearn.linear_model import LogisticRegression
model = LogisticRegression(multi_class='multinomial', max_iter=2000)

model.fit(X,y)

In [ ]:
X_test = vectorizer.transform(test["prompt_and_response"].astype(str))
prediction = model.predict(X_test)
probs = model.predict_proba(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'winner_model_a': probs[:,0],
    'winner_model_b': probs[:,1],
    'winner_tie': probs[:,2]
})
print(submission.head())

submission.to_csv("submission.csv", index=False)
